# 04 - Whitepaper Figures
Generate all publication-quality charts and the summary report.

In [ ]:
import sys
sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'white'

In [ ]:
# Load all data and analysis results
from src.polymarket.market_discovery import load_discovered_markets
from src.polymarket.timeseries import TimeseriesFetcher
from src.freight.scraper import fetch_all_freight_indexes
from src.freight.normalize import prepare_freight_panel
from src.analysis.events import EventDetector
from src.analysis.correlation import CorrelationAnalyser, event_study
from src.analysis.impact_mapper import ImpactMapper

markets_df = load_discovered_markets()
fetcher = TimeseriesFetcher()
timeseries = fetcher.fetch_all(markets_df)
freight_raw = fetch_all_freight_indexes(use_synthetic_fallback=True)
freight_data = prepare_freight_panel(freight_raw)

detector = EventDetector()
all_events = detector.detect_all(timeseries, markets_df)
events_df = detector.to_dataframe(all_events)

analyser = CorrelationAnalyser()
xcorr_results = analyser.run_cross_correlations(timeseries, freight_data, markets_df)
xcorr_df = analyser.xcorr_to_dataframe(xcorr_results)
granger_results = analyser.run_granger_tests(timeseries, freight_data, markets_df)
granger_df = analyser.granger_to_dataframe(granger_results)

mapper = ImpactMapper()
assessments = mapper.generate_assessments(all_events, markets_df, xcorr_results)

print('All data and analysis loaded.')
print(f'Markets: {len(timeseries)}, Events: {len(all_events)}, XCorr pairs: {len(xcorr_results)}')

## Chart 1: Dual-axis Overlay (Hero Chart)

In [ ]:
from src.visualization.charts import (
    plot_dual_axis_overlay,
    plot_cross_correlation,
    plot_event_study,
    plot_correlation_heatmap,
    plot_annotated_timeline,
)

if xcorr_results:
    best = max(xcorr_results, key=lambda r: abs(r.peak_correlation))
    poly_df = timeseries.get(best.market_id)
    freight_df = freight_data.get(best.freight_index)
    mkt_events = (
        events_df[events_df['market_id'].astype(str) == str(best.market_id)]
        if not events_df.empty else pd.DataFrame()
    )
    event_ts = (
        list(pd.to_datetime(mkt_events['timestamp']))
        if not mkt_events.empty else []
    )
    if poly_df is not None and freight_df is not None:
        fig = plot_dual_axis_overlay(
            poly_df, freight_df,
            best.market_title, best.freight_index,
            event_timestamps=event_ts,
            filename_stem='fig1_dual_axis_hero',
        )
        plt.show()

## Chart 2: Cross-Correlation Plot

In [ ]:
if xcorr_results:
    best = max(xcorr_results, key=lambda r: abs(r.peak_correlation))
    fig = plot_cross_correlation(
        best.lags, best.correlations, best.p_values,
        best.market_title, best.freight_index,
        best.peak_lag, best.peak_correlation,
        filename_stem='fig2_xcorr_best',
    )
    plt.show()
    print(f'Interpretation: {best.interpretation}')

## Chart 3: Event Study

In [ ]:
event_study_results = []
for r in xcorr_results[:3]:
    poly_df = timeseries.get(r.market_id)
    freight_df = freight_data.get(r.freight_index)
    if poly_df is None or freight_df is None:
        continue
    mkt_events = (
        events_df[events_df['market_id'].astype(str) == str(r.market_id)]
        if not events_df.empty else pd.DataFrame()
    )
    if mkt_events.empty:
        continue
    event_ts_list = list(pd.to_datetime(mkt_events['timestamp']))
    es = event_study(
        r.market_id, r.market_title,
        event_ts_list, freight_df, r.freight_index,
    )
    if es is not None:
        event_study_results.append(es)
        fig = plot_event_study(
            es.event_window, es.mean_freight_change,
            es.ci_lower, es.ci_upper,
            es.baseline_change,
            es.market_title, es.freight_index, es.n_events,
            filename_stem=f'fig3_event_study_{r.market_id[:8]}',
        )
        plt.show()
        print(f'CAR (cumulative abnormal return): {es.cumulative_abnormal_return:.2f}%')

## Chart 4: Correlation Heatmap

In [ ]:
if not xcorr_df.empty:
    fig = plot_correlation_heatmap(xcorr_df, filename_stem='fig4_heatmap')
    plt.show()

## Chart 5: Annotated Timeline

In [ ]:
if xcorr_results and not events_df.empty:
    best = xcorr_results[0]
    poly_df = timeseries.get(best.market_id)
    freight_df = freight_data.get(best.freight_index)
    mkt_events = events_df[events_df['market_id'].astype(str) == str(best.market_id)]
    if poly_df is not None and freight_df is not None and not mkt_events.empty:
        fig = plot_annotated_timeline(
            poly_df, freight_df, mkt_events,
            best.market_title, best.freight_index,
            filename_stem='fig5_timeline',
        )
        plt.show()

## Generate Summary Report

In [ ]:
from pathlib import Path
from datetime import datetime

report_lines = []
report_lines.append('# Polymarket Supply Chain Intelligence - Analysis Report')
report_lines.append(f'
*Generated: {datetime.now().strftime("%Y-%m-%d %H:%M UTC")}*
')
report_lines.append('---
')
report_lines.append('## Executive Summary')
report_lines.append(f'- **Markets discovered:** {len(markets_df)}')
report_lines.append(f"- **Active markets:** {(markets_df['status'] == 'active').sum()}")
report_lines.append(f"- **Historical (closed) markets:** {(markets_df['status'] == 'closed').sum()}")
report_lines.append(f'- **Markets with timeseries data:** {len(timeseries)}')
report_lines.append(f'- **Significant probability shift events detected:** {len(all_events)}')
report_lines.append(f'- **Market-freight pairings analysed:** {len(xcorr_results)}')
report_lines.append(f"- **Statistically significant correlations (p < 0.05):** {xcorr_df['is_significant'].sum() if not xcorr_df.empty else 0}")
report_lines.append(f'- **Granger causality tests run:** {len(granger_results)}')
report_lines.append(f"- **Significant Granger results:** {granger_df['is_significant'].sum() if not granger_df.empty else 0}")
report_lines.append('
---
')
report_lines.append('## Key Findings')
report_lines.append('### Cross-Correlation Results')

if not xcorr_df.empty:
    for _, row in xcorr_df.sort_values('peak_correlation', ascending=False).head(5).iterrows():
        leads = 'Polymarket LEADS' if row['polymarket_leads'] else 'Freight leads'
        sig = 'significant' if row['is_significant'] else 'not significant'
        report_lines.append(f"- **{row['market_title'][:50]}** x {row['freight_index']}: r={row['peak_correlation']:.3f}, lag={row['peak_lag_days']}d ({leads}, {sig})")

report_lines.append('
### Granger Causality')

if not granger_df.empty:
    for _, row in granger_df.sort_values('min_p_value').head(5).iterrows():
        sig = 'SIGNIFICANT' if row['is_significant'] else 'not significant'
        report_lines.append(f"- **{row['market_title'][:50]}** -> {row['freight_index']}: p={row['min_p_value']:.4f} at lag={row['best_lag']}d ({sig})")

report_lines.append('
---
')
report_lines.append(mapper.generate_report_section(assessments, top_n=5))
report_lines.append('
---
')
report_lines.append('## Figures Generated')

figures_dir = Path('../output/figures')
if figures_dir.exists():
    for fig_file in sorted(figures_dir.glob('*.png')):
        report_lines.append(f'- ')

report_lines.append('
---
')
report_lines.append('## Caveats and Limitations')
report_lines.append('1. **Synthetic freight data**: Where live freight data was unavailable, synthetic data was generated for pipeline validation. Final analysis requires real BDI/FBX data.')
report_lines.append('2. **Correlation != causation**: Statistical relationships documented here are consistent with the leading indicator thesis but require further validation.')
report_lines.append('3. **Sample size**: Some pairings have limited overlapping observations, reducing statistical power.')
report_lines.append('4. **Market selection bias**: Markets were selected by keyword matching; some relevant markets may be missed.')
report_lines.append('5. **Data gaps**: Some Polymarket timeseries have gaps that could affect correlation estimates.')
report_lines.append('
---
')
report_lines.append('*Report generated by Polymarket SCM Intelligence MVP pipeline.*')

report_text = '
'.join(report_lines)
report_path = Path('../output/report.md')
report_path.parent.mkdir(exist_ok=True)
report_path.write_text(report_text, encoding='utf-8')
print(f'Report saved to {report_path.resolve()}')
print('
Report preview (first 2000 chars):')
print(report_text[:2000])

## Complete
All whitepaper figures and the summary report have been generated.

Figures are saved in output/figures/ (PNG + SVG). Report is at output/report.md.